In [ ]:
# =========================
# IMPORTURI & CONFIG
# =========================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from packaging import version

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SelectKBest, mutual_info_regression, SelectFromModel
from sklearn.linear_model import LinearRegression, Ridge, Lasso, QuantileRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("Importuri OK — scikit-learn", version.parse(__import__("sklearn").__version__))

TRAIN_PATH = Path("train_cars_listings.csv")
VAL_PATH = Path("val_cars_listings.csv")
TARGET = "pret"


In [ ]:
# =========================
# ÎNCĂRCARE DATE + FEATURE ENGINEERING DE BAZĂ
# =========================
def load_autovit(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # curățăm whitespace și standardizăm numele coloanelor
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    # convertim la tipuri adecvate
    date_cols = [c for c in df.columns if "data" in c]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    if "anul_fabricatiei" in df.columns:
        df["vechime"] = pd.Timestamp("now").year - df["anul_fabricatiei"]
    if "km" in df.columns:
        df["log_km"] = np.log1p(df["km"].clip(lower=0))
    if "pret" in df.columns:
        df["log_pret"] = np.log1p(df["pret"].clip(lower=0))
    return df

train_df = load_autovit(TRAIN_PATH)
val_df = load_autovit(VAL_PATH)
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
train_df.head()


## PART 1 - Exploratory Data Analysis (EDA)


### Analiza 1 - Structura setului și valori lipsă

Setul Autovit conține multe coloane heterogene, inclusiv liste și câmpuri text. Trebuie să știm ce tipuri predomină și unde apar lipsuri pentru a planifica imputarea.

`info()`, statistici numerice, procentul de valori lipsă per coloană

In [ ]:
train_df.info()

num_summary = train_df.select_dtypes(include=[np.number]).describe().T
num_summary

missing_pct = train_df.isna().mean().sort_values(ascending=False)
missing_top = missing_pct[missing_pct > 0].head(20)
plt.figure(figsize=(10,5))
missing_top.sort_values().plot(kind='barh', color='#d95f02')
plt.title('Top 20 coloane cu valori lipsă (procent)')
plt.xlabel('Procent valori lipsă')
plt.show()

missing_pct.head(20)


### Analiza 2 - Distribuția target-ului `pret`

Prețurile auto sunt de obicei heavy-tailed; trebuie să verific dacă transformarea logaritmică este justificată și ce metrici să raportăm (MAE vs. RMSE).

Histogramă + boxplot pentru `pret` și `log_pret`, plus cuantile relevante.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].hist(train_df[TARGET].dropna(), bins=60, color='#4c78a8', edgecolor='white')
axes[0].set_title('Distribuție pret brut')
axes[0].set_xlabel('pret (EUR)')
axes[1].hist(train_df['log_pret'].dropna(), bins=60, color='#9c755f', edgecolor='white')
axes[1].set_title('Distribuție log_pret')
axes[1].set_xlabel('log1p(pret)')
plt.tight_layout(); plt.show()

pd.DataFrame({
    'metric': ['median', 'p90', 'p95', 'max'],
    'pret': [train_df[TARGET].median(), train_df[TARGET].quantile(0.9), train_df[TARGET].quantile(0.95), train_df[TARGET].max()],
    'log_pret': [train_df['log_pret'].median(), train_df['log_pret'].quantile(0.9), train_df['log_pret'].quantile(0.95), train_df['log_pret'].max()]
})


### Analiza 3 - Relația dintre preț și variabile numerice

Identificăm cei mai puternici predicatori numeric (km, vechime, putere etc.) pentru a prioritiza transformări/discretizări.

Matrice de corelație Pearson + pairplot pentru subset numeric.


In [ ]:
num_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns if c not in {"log_pret"}]
num_cols = [c for c in num_cols if train_df[c].notnull().mean() > 0.6]
subset = num_cols[:8]
plt.figure(figsize=(10,8))
sns.heatmap(train_df[subset + [TARGET]].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Corelații numerice (subset)')
plt.show()

sns.pairplot(train_df, vars=[TARGET, 'km', 'vechime', 'putere'] if set(['km','vechime','putere']).issubset(train_df.columns) else [TARGET])


### Analiza 4 - Impactul categoriilor (marca, combustibil, transmisie)

Prețul variază semnificativ în funcție de marca și tipul mașinii; graficele boxplot ajută la justificarea encodării categorice și eventual la gruparea valorilor rare.

Boxplot pentru `pret` în funcție de `marca`, `combustibil`, `transmisie` (top 10 categorii).


In [ ]:
for col in ["marca", "combustibil", "transmisie"]:
    if col not in train_df.columns:
        continue
    top_vals = train_df[col].value_counts().head(10).index
    plt.figure(figsize=(12,4))
    sns.boxplot(data=train_df[train_df[col].isin(top_vals)], x=col, y=TARGET)
    plt.title(f"Distribuția pret pentru top 10 {col}")
    plt.xticks(rotation=30)
    plt.show()



**Concluzii EDA:**
- Setul are multe coloane cu peste 30% lipsuri (ex. `versiune`, dotări), deci imputarea este obligatorie; pentru liste rare vom folosi abordări simple (flag lipsă + fallback `None`).
- Distribuția `pret` este extrem de asimetrică, log-transformarea stabilizează varianța => vom optimiza pe valori reale dar vom monitoriza și `log_pret`.
- Variabilele numerice `km`, `vechime`, `putere` arată corelații moderate cu prețul, ceea ce justifică standardizarea și, eventual, discretizarea (ex. KBins pe `km`).
- Marca/combustibilul influențează puternic medianele, deci One-Hot Encoder cu handling pentru valori rare este potrivit.



## PART 2 - Preprocesare (imputare, standardizare, encodare, discretizare, selecție a atributelor)

Strategie:
1. Separăm coloanele numerice/categorice și eliminăm câmpurile redundant (ex. `nume`, coloanele listă greu de procesat).
2. Construim un split train/valid pe 80/20 (aleator dar cu `random_state` fix, deoarece datele nu sunt temporale).
3. Pipeline:
   - Numerice: imputare cu mediană + standardizare.
   - Categorical high-cardinality: One-Hot cu `handle_unknown='ignore'` + reducerea valorilor rare la `Other`.
   - Discretizare: `KBinsDiscretizer` pentru `km` și `vechime` (5 quantile) pentru a surprinde efecte neliniare.
4. Selecție de atribute cu `SelectKBest (mutual_info)` și `SelectFromModel (Lasso)` pe matricea procesată.



In [ ]:
DROP_COLS = [
    c for c in ["nume", "descriere", "dotari", "audio_si_tehnologie", "confort_si_echipamente_optionale",
                 "electronice_si_sisteme_de_asistenta", "siguranta", "vehicule_electrice", "versiune"]
    if c in train_df.columns
]

used_cols = [c for c in train_df.columns if c not in DROP_COLS]
print("Coloane folosite:", len(used_cols))

model_df = train_df[used_cols].copy()
model_df = model_df[model_df[TARGET].notnull()]  # target necesar

numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in model_df.columns if c not in numeric_cols + [TARGET, 'log_pret']]

print("Numeric:", len(numeric_cols), "Categorice:", len(categorical_cols))


In [ ]:
X = model_df.drop(columns=[TARGET])
y = model_df[TARGET]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=None
)
print("Split sizes:", X_train.shape, X_valid.shape)



In [ ]:
NUMERIC_STD = [c for c in numeric_cols if c not in ["log_pret"]]
NUMERIC_STD = [c for c in NUMERIC_STD if c in X_train.columns]
BIN_COLS = [c for c in ["km", "vechime"] if c in X_train.columns]
CAT_COLS = [c for c in categorical_cols if X_train[c].dtype == object]
ORDINAL_COLS = []  # dacă există ordinale specifice (ex. cutie manual/automat), pot fi mutate aici

class RareLabelEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=0.01):
        self.min_freq = min_freq
        self.frequent_values_ = {}
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X, columns=getattr(X, 'columns', None))
        for col in X_df.columns:
            freqs = X_df[col].value_counts(normalize=True)
            self.frequent_values_[col] = freqs[freqs >= self.min_freq].index.tolist()
        return self
    def transform(self, X):
        X_df = pd.DataFrame(X, columns=getattr(X, 'columns', None))
        for col in X_df.columns:
            allowed = set(self.frequent_values_.get(col, []))
            X_df[col] = np.where(X_df[col].isin(allowed), X_df[col], 'Other')
        return X_df.values if isinstance(X, np.ndarray) else X_df

rare_encoder = RareLabelEncoder(min_freq=0.01)


In [ ]:
numeric_pipeline = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("rare", rare_encoder),
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

bin_pipeline = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("kbins", KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="quantile"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_STD),
        ("cat", categorical_pipeline, CAT_COLS),
        ("bins", bin_pipeline, BIN_COLS)
    ],
    remainder="drop",
    sparse_threshold=0.0
)

preprocess.fit(X_train)
X_train_proc = preprocess.transform(X_train)
X_valid_proc = preprocess.transform(X_valid)
print("Preprocessed shapes:", X_train_proc.shape, X_valid_proc.shape)


In [ ]:
# Feature selection + Ridge

sel_results = []

p = X_train_proc.shape[1]
ks = sorted({30, 60, 90, 120, int(0.3 * p), int(0.5 * p)})
ks = [k for k in ks if 20 <= k <= p]

best_rmse = np.inf
best_cfg_name = None
best_selector = None
best_Xtr_sel = None
best_Xva_sel = None

# -------------------------
# 1) SelectKBest + mutual_info_regression
# -------------------------
for k in ks:
    selector = SelectKBest(mutual_info_regression, k=k)
    Xtr_sel = selector.fit_transform(X_train_proc, y_train)
    Xva_sel = selector.transform(X_valid_proc)

    ridge = Ridge(alpha=0.1, random_state=RANDOM_STATE)
    ridge.fit(Xtr_sel, y_train)
    preds = ridge.predict(Xva_sel)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))

    cfg_name = f"SelectKBest k={k}"
    sel_results.append({
        "config": cfg_name,
        "n_features": k,
        "Valid_RMSE": rmse
    })

    # ținem minte cea mai bună configurație
    if rmse < best_rmse:
        best_rmse = rmse
        best_cfg_name = cfg_name
        best_selector = selector
        best_Xtr_sel = Xtr_sel
        best_Xva_sel = Xva_sel

# -------------------------
# 2) SelectFromModel cu Lasso (mai puțin agresiv ca timp)
# -------------------------
lasso_alphas = [0.0005, 0.001, 0.005, 0.01]  # am redus un pic lista

for alpha in lasso_alphas:
    lasso = Lasso(
        alpha=alpha,
        max_iter=3000,
        tol=1e-3,         # convergență mai „relaxată”
        random_state=RANDOM_STATE
    )

    sfm = SelectFromModel(lasso, threshold="mean")
    Xtr_sel = sfm.fit_transform(X_train_proc, y_train)
    Xva_sel = sfm.transform(X_valid_proc)

    ridge = Ridge(alpha=0.1, random_state=RANDOM_STATE)
    ridge.fit(Xtr_sel, y_train)
    preds = ridge.predict(Xva_sel)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))

    cfg_name = f"SelectFromModel Lasso α={alpha}"
    sel_results.append({
        "config": cfg_name,
        "n_features": Xtr_sel.shape[1],
        "Valid_RMSE": rmse
    })

    # actualizăm best dacă e mai bun
    if rmse < best_rmse:
        best_rmse = rmse
        best_cfg_name = cfg_name
        best_selector = sfm
        best_Xtr_sel = Xtr_sel
        best_Xva_sel = Xva_sel

# -------------------------
# Rezultate + transformarea câștigătoare
# -------------------------

sel_df = pd.DataFrame(sel_results).sort_values("Valid_RMSE").reset_index(drop=True)

# păstrăm DOAR cea mai bună transformare
sel_transforms = {}
if best_cfg_name is not None:
    sel_transforms[best_cfg_name] = (best_selector, best_Xtr_sel, best_Xva_sel)

sel_df


In [ ]:
best_sel = sel_df.iloc[0]
best_name = best_sel['config']
best_selector, X_train_sel, X_valid_sel = sel_transforms[best_name]
print(f"Selector ales: {best_name} | n_features={X_train_sel.shape[1]} | RMSE_valid={best_sel['Valid_RMSE']:.2f}")


## PART 3 - Modele + tuning + evaluare

Vom reutiliza `preprocess` + `best_selector` într-un `Pipeline` și vom evalua modelele cerute. Pentru fiecare: `TimeSeriesSplit` nu este relevant pe date i.i.d., dar păstrăm `KFold` echivalent (folosim `TimeSeriesSplit` pentru consistență). Metricile raportate: MSE, MAE, RMSE, R² pe validare + MSE mediu pe CV.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate_model(name, estimator, param_grid=None, n_iter=20):
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    if param_grid:
        search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True
        )
        search.fit(X_train, y_train)
        best_estimator = search.best_estimator_
        cv_mse = -search.best_score_
        best_params = search.best_params_
    else:
        pipeline.fit(X_train, y_train)
        best_estimator = pipeline
        cv_scores = []
        for tr_idx, te_idx in cv.split(X_train):
            X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
            y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]
            pipeline.fit(X_tr, y_tr)
            preds = pipeline.predict(X_te)
            cv_scores.append(mean_squared_error(y_te, preds))
        cv_mse = np.mean(cv_scores)
        best_params = {}

    y_pred = best_estimator.predict(X_valid)
    mse = mean_squared_error(y_valid, y_pred)
    mae = mean_absolute_error(y_valid, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_valid, y_pred)
    return {
        "model": name,
        "CV_MSE": cv_mse,
        "Valid_MSE": mse,
        "Valid_MAE": mae,
        "Valid_RMSE": rmse,
        "Valid_R2": r2,
        "best_estimator": best_estimator,
        "best_params": best_params
    }

best_models = {}
results_rows = []


In [ ]:
lin_configs = [
    ("LinearRegression", LinearRegression()),
    ("LinearRegression (no_intercept)", LinearRegression(fit_intercept=False)),
    ("LinearRegression (positive)", LinearRegression(positive=True))
]
lin_rows = []
for name, est in lin_configs:
    res = evaluate_model(name, est)
    lin_rows.append(res)

lin_df = pd.DataFrame([{k:v for k,v in r.items() if k not in {"best_estimator","best_params"}} for r in lin_rows]).sort_values("Valid_MSE")
lin_df

best_lin = lin_rows[np.argmin([r["Valid_MSE"] for r in lin_rows])]
best_models["LinearRegression"] = best_lin
results_rows.append({k:v for k,v in best_lin.items() if k not in {"best_estimator","best_params"}})
print("Best linear config:", best_lin["model"])


In [ ]:
svr_param_dist = {
    "model__kernel": ["linear", "rbf"],
    "model__C": [0.3, 1, 3, 10, 30, 100],
    "model__epsilon": [0.05, 0.1, 0.2, 0.3],
    "model__gamma": ["scale", 0.001, 0.01, 0.03, 0.1]
}
svr_result = evaluate_model("SVR", SVR(), param_grid=svr_param_dist, n_iter=30)
best_models["SVR"] = svr_result
results_rows.append({k:v for k,v in svr_result.items() if k not in {"best_estimator","best_params"}})
print("SVR best params:", svr_result["best_params"])


In [ ]:
rf_param_dist = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [None, 8, 12, 16, 24],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 4, 6],
    "model__bootstrap": [True, False]
}
rf_result = evaluate_model("RandomForest", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), param_grid=rf_param_dist, n_iter=40)
best_models["RandomForest"] = rf_result
results_rows.append({k:v for k,v in rf_result.items() if k not in {"best_estimator","best_params"}})
print("RF best params:", rf_result["best_params"])


In [ ]:
gb_param_dist = {
    "model__n_estimators": [200, 300, 400, 600],
    "model__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", 0.5, None]
}
gb_result = evaluate_model("GBR (squared)", GradientBoostingRegressor(loss="squared_error", random_state=RANDOM_STATE), param_grid=gb_param_dist, n_iter=30)
best_models["GBR_squared"] = gb_result
results_rows.append({k:v for k,v in gb_result.items() if k not in {"best_estimator","best_params"}})
print("GBR squared best params:", gb_result["best_params"])


In [ ]:
quantile_results = {}
quantile_preds = {}
quantile_param_dist = {
    "model__n_estimators": [300, 400, 600],
    "model__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", 0.5, None]
}
for alpha in [0.10, 0.50, 0.90]:
    estimator = GradientBoostingRegressor(loss="quantile", alpha=alpha, random_state=RANDOM_STATE)
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=quantile_param_dist,
        n_iter=25,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True
    )
    search.fit(X_train, y_train)
    best_est = search.best_estimator_
    preds = best_est.predict(X_valid)
    quantile_results[alpha] = {
        "best_estimator": best_est,
        "best_params": search.best_params_,
        "CV_MSE": -search.best_score_,
        "Valid_MSE": mean_squared_error(y_valid, preds),
        "Valid_MAE": mean_absolute_error(y_valid, preds),
        "Valid_R2": r2_score(y_valid, preds)
    }
    quantile_preds[alpha] = preds
    print(f"alpha={alpha} -> {search.best_params_}")

median_metrics = quantile_results[0.50]
coverage = np.mean((y_valid.values >= quantile_preds[0.10]) & (y_valid.values <= quantile_preds[0.90]))
best_models["GBR_quantile"] = {
    "model": "GBR quantile (median)",
    "best_estimator": quantile_results[0.50]["best_estimator"],
    "best_params": quantile_results[0.50]["best_params"],
    "Valid_MSE": median_metrics["Valid_MSE"],
    "Valid_MAE": median_metrics["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_metrics["Valid_MSE"]),
    "Valid_R2": median_metrics["Valid_R2"],
    "CV_MSE": median_metrics["CV_MSE"],
    "coverage": coverage
}
results_rows.append({
    "model": "GBR quantile (median)",
    "CV_MSE": median_metrics["CV_MSE"],
    "Valid_MSE": median_metrics["Valid_MSE"],
    "Valid_MAE": median_metrics["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_metrics["Valid_MSE"]),
    "Valid_R2": median_metrics["Valid_R2"]
})
print(f"Coverage [0.10,0.90] = {coverage:.1%}")


In [ ]:
qr_results = {}
alpha_grid = [0.0, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.3, 1.0, 3.0]
for tau in [0.10, 0.50, 0.90]:
    estimator = QuantileRegressor(quantile=tau, solver="highs", fit_intercept=True)
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        pipeline,
        param_distributions={"model__alpha": alpha_grid},
        n_iter=min(8, len(alpha_grid)),
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True
    )
    search.fit(X_train, y_train)
    best_est = search.best_estimator_
    preds = best_est.predict(X_valid)
    qr_results[tau] = {
        "best_estimator": best_est,
        "best_params": search.best_params_,
        "CV_MSE": -search.best_score_,
        "Valid_MSE": mean_squared_error(y_valid, preds),
        "Valid_MAE": mean_absolute_error(y_valid, preds),
        "Valid_R2": r2_score(y_valid, preds)
    }
    print(f"τ={tau} -> {search.best_params_}")

median_qr = qr_results[0.50]
coverage_qr = np.mean((y_valid.values >= qr_results[0.10]["best_estimator"].predict(X_valid)) &
                      (y_valid.values <= qr_results[0.90]["best_estimator"].predict(X_valid)))

best_models["QuantileRegressor"] = {
    "model": "QuantileRegressor (median)",
    "best_estimator": median_qr["best_estimator"],
    "best_params": median_qr["best_params"],
    "Valid_MSE": median_qr["Valid_MSE"],
    "Valid_MAE": median_qr["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_qr["Valid_MSE"]),
    "Valid_R2": median_qr["Valid_R2"],
    "CV_MSE": median_qr["CV_MSE"],
    "coverage": coverage_qr
}
results_rows.append({
    "model": "QuantileRegressor (median)",
    "CV_MSE": median_qr["CV_MSE"],
    "Valid_MSE": median_qr["Valid_MSE"],
    "Valid_MAE": median_qr["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_qr["Valid_MSE"]),
    "Valid_R2": median_qr["Valid_R2"]
})
print(f"QuantileRegressor coverage [0.10,0.90] = {coverage_qr:.1%}")


In [ ]:
results_df = pd.DataFrame(results_rows).set_index("model").sort_values("Valid_MSE")
results_df


## PART 4 - Evaluare pe `val_cars_listings.csv`

După alegerea modelelor, reantrenăm pe întregul set `train_df` și raportăm metricile pe setul de validare final (`val_df`), care conține ținta `pret`. În plus, exportăm predicțiile modelului cu cel mai bun MSE într-un CSV și păstrăm benzile de cuantile.


In [ ]:
val_processed = val_df[used_cols].copy()
X_val_final = val_processed.drop(columns=[TARGET])
y_val_final = val_processed[TARGET]
print("Val final shape:", X_val_final.shape)

X_full = pd.concat([X_train, X_valid])
y_full = pd.concat([y_train, y_valid])


In [ ]:
eval_rows = []

def compute_eval(label, estimator):
    est = clone(estimator)
    est.fit(X_full, y_full)
    preds = est.predict(X_val_final)
    mse = mean_squared_error(y_val_final, preds)
    mae = mean_absolute_error(y_val_final, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val_final, preds)
    eval_rows.append({
        "model": label,
        "Eval_MSE": mse,
        "Eval_MAE": mae,
        "Eval_RMSE": rmse,
        "Eval_R2": r2
    })
    return preds

for key, info in best_models.items():
    if key in {"GBR_quantile", "QuantileRegressor"}:
        continue
    compute_eval(info.get("model", key), info["best_estimator"])

quantile_eval_preds = {}
for alpha in [0.10, 0.50, 0.90]:
    est = clone(quantile_results[alpha]["best_estimator"])
    est.fit(X_full, y_full)
    quantile_eval_preds[alpha] = est.predict(X_val_final)

coverage_eval_gb = np.mean((y_val_final.values >= quantile_eval_preds[0.10]) & (y_val_final.values <= quantile_eval_preds[0.90]))
compute_eval("GBR quantile (median)", quantile_results[0.50]["best_estimator"])

qr_eval_preds = {}
for tau in [0.10, 0.50, 0.90]:
    est = clone(qr_results[tau]["best_estimator"])
    est.fit(X_full, y_full)
    qr_eval_preds[tau] = est.predict(X_val_final)

coverage_eval_qr = np.mean((y_val_final.values >= qr_eval_preds[0.10]) & (y_val_final.values <= qr_eval_preds[0.90]))
compute_eval("QuantileRegressor (median)", qr_results[0.50]["best_estimator"])

print(f"GBR quantile coverage val: {coverage_eval_gb:.1%}")
print(f"QuantileRegressor coverage val: {coverage_eval_qr:.1%}")

eval_df = pd.DataFrame(eval_rows).set_index("model").sort_values("Eval_MSE")
eval_df


In [ ]:
best_eval_label = eval_df.index[0]
print("Cel mai bun model pe val:", best_eval_label)

if best_eval_label.startswith("GBR quantile"):
    estimator = quantile_results[0.50]["best_estimator"]
elif best_eval_label.startswith("QuantileRegressor"):
    estimator = qr_results[0.50]["best_estimator"]
else:
    estimator = None
    for info in best_models.values():
        if info.get("model") == best_eval_label:
            estimator = info["best_estimator"]
            break
    if estimator is None:
        estimator = best_models["RandomForest"]["best_estimator"]

estimator = clone(estimator)
estimator.fit(X_full, y_full)
preds_val = estimator.predict(X_val_final)
pred_df = pd.DataFrame({
    "id": val_processed.index,
    "pret_real": y_val_final.values,
    "pret_pred": preds_val
})
pred_path = Path("pred_autovit_best_model.csv")
pred_df.to_csv(pred_path, index=False)
print("Predicții salvate în", pred_path.resolve())


## Concluzii

- **EDA** evidențiază lipsuri masive și variația prețului pe categorii; acest lucru motivează pipeline-ul de imputare + rare-label + OHE.
- **Preprocesare**: numerice standardizate + discretizare `km`/`vechime`, categorice tratate prin RareLabelEncoder -> OHE, selecție `SelectKBest` / `SelectFromModel` pentru a reduce dimensionalitatea.
- **Modele**: toate cerute în enunț sunt evaluate cu `KFold(5)` și `RandomizedSearchCV`.
- **Predicții finale**: `pred_autovit_best_model.csv` conține valori reale vs. estimate 
- **Modele pe cuantile**: raportează acoperiri pe validare și val

